# The Transformer, end to end
1. One sentence pair through a base-size transformer (d_model = 512, h = 8, d_ff = 2048, N = 6): every shape, and the parameter counts.
2. The warm-up learning-rate schedule of §5.3.
3. Training time and cost arithmetic from §5.2, §6.1 and Table 2.
4. Label smoothing (§5.4) on a 4-word vocabulary, with ε = 0.1.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""          # shapes only: CPU is enough
import numpy as np
import pandas as pd
import keras
keras.utils.set_random_seed(0)
np.set_printoptions(precision=4, suppress=True)

## 1. One sentence pair through the base model
Random weights: only shapes and parameter counts matter here. Encoder input "we're friends ." (3 tokens); decoder input "<start> nous sommes amis ." (5 positions, shifted right). Token ids are made up. One shared vocabulary of 37,000 tokens (§5.1).

In [ ]:
d, h, d_ff, N, V = 512, 8, 2048, 6, 37000

class EncoderBlock(keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self.mha = keras.layers.MultiHeadAttention(num_heads=h, key_dim=d // h)
        self.norm1, self.norm2 = keras.layers.LayerNormalization(), keras.layers.LayerNormalization()
        self.ffn1, self.ffn2 = keras.layers.Dense(d_ff, activation="relu"), keras.layers.Dense(d)
    def call(self, x):
        z = self.norm1(x + self.mha(x, x))
        return self.norm2(z + self.ffn2(self.ffn1(z)))

class DecoderBlock(keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self.self_mha = keras.layers.MultiHeadAttention(num_heads=h, key_dim=d // h)
        self.cross_mha = keras.layers.MultiHeadAttention(num_heads=h, key_dim=d // h)
        self.norms = [keras.layers.LayerNormalization() for _ in range(3)]
        self.ffn1, self.ffn2 = keras.layers.Dense(d_ff, activation="relu"), keras.layers.Dense(d)
    def call(self, x, h_enc):
        x = self.norms[0](x + self.self_mha(x, x, use_causal_mask=True))
        x = self.norms[1](x + self.cross_mha(x, h_enc))          # queries from x, keys and values from h_enc
        return self.norms[2](x + self.ffn2(self.ffn1(x)))

def positional_encoding(n, d):
    pos, i = np.arange(n)[:, None], np.arange(0, d, 2)[None, :]
    pe = np.zeros((n, d))
    pe[:, 0::2], pe[:, 1::2] = np.sin(pos / 10000 ** (i / d)), np.cos(pos / 10000 ** (i / d))
    return pe.astype("float32")

embed = keras.layers.Embedding(V, d)                 # shared: encoder, decoder and output layer (§3.4)
enc_blocks = [EncoderBlock() for _ in range(N)]
dec_blocks = [DecoderBlock() for _ in range(N)]

src = np.array([[11, 254, 4]])                       # we're friends .
tgt_in = np.array([[1, 21, 85, 250, 4]])             # <start> nous sommes amis .

steps = []
e = embed(src) * np.sqrt(d);                 steps.append(("encoder: embeddings x sqrt(d_model)", e.shape))
x = e + positional_encoding(3, d);           steps.append(("encoder: + positional encoding", x.shape))
for b in enc_blocks: x = b(x)
h_enc = x;                                   steps.append(("encoder: after 6 blocks (H_enc)", h_enc.shape))
y = embed(tgt_in) * np.sqrt(d) + positional_encoding(5, d); steps.append(("decoder: shifted input + positions", y.shape))
for b in dec_blocks: y = b(y, h_enc)
steps.append(("decoder: after 6 blocks", y.shape))
W = embed.embeddings                          # output layer reuses the embedding matrix (§3.4)
logits = keras.ops.matmul(y, keras.ops.transpose(W)); steps.append(("logits", logits.shape))
probs = keras.ops.softmax(logits, axis=-1);   steps.append(("softmax probabilities", probs.shape))
for s, shp in steps: print(f"{s:40s} {tuple(shp)}")
print("each row of probabilities sums to", np.round(np.asarray(probs).sum(-1), 4))

In [ ]:
# Shapes inside one encoder block, sub-layer by sub-layer
b = enc_blocks[0]; x0 = e + positional_encoding(3, d)
z = b.mha(x0, x0); zn = b.norm1(x0 + z); hid = b.ffn1(zn); out = b.norm2(zn + b.ffn2(hid))
for s, t in [("X", x0), ("multi-head attention Z", z), ("add & norm", zn), ("FFN hidden", hid), ("block output", out)]:
    print(f"{s:25s} {tuple(t.shape)}")
# attention weights: one 3x3 table per head
_, w = b.mha(x0, x0, return_attention_scores=True)
print("encoder self-attention weights:", tuple(w.shape))
_, wc = dec_blocks[0].cross_mha(embed(tgt_in), h_enc, return_attention_scores=True)
print("cross-attention weights:", tuple(wc.shape))
_, wm = dec_blocks[0].self_mha(embed(tgt_in), embed(tgt_in), use_causal_mask=True, return_attention_scores=True)
print("masked self-attention weights:", tuple(wm.shape))

In [ ]:
enc_p, dec_p, emb_p = enc_blocks[0].count_params(), dec_blocks[0].count_params(), embed.count_params()
print("one encoder block:", f"{enc_p:,}")
print("one decoder block:", f"{dec_p:,}")
print("6 + 6 blocks:", f"{N * (enc_p + dec_p):,}")
print("shared embedding 37,000 x 512:", f"{emb_p:,}")
total = N * (enc_p + dec_p) + emb_p
print("total:", f"{total:,}")
assert enc_p == 3_152_384 and dec_p == 4_204_032

## 2. The warm-up learning-rate schedule (§5.3, eq. 3)
lrate = d_model^-0.5 · min(step^-0.5, step · warmup^-1.5), with d_model = 512, warmup_steps = 4000.

In [ ]:
def lrate(step, d_model=512, warmup=4000):
    step = np.asarray(step, dtype=float)
    return d_model ** -0.5 * np.minimum(step ** -0.5, step * warmup ** -1.5)

for s in [1, 100, 1000, 2000, 4000, 8000, 16000, 50000, 100000]:
    print(f"step {s:>7,}: {lrate(s):.3e}")
steps_ = np.arange(1, 100_001)
lr = lrate(steps_)
print("peak at step", steps_[lr.argmax()], f"value {lr.max():.3e}")
print("peak / value at 100,000:", round(lr.max() / lrate(100_000), 3))
print("value at 2000 / value at 1000:", round(lrate(2000) / lrate(1000), 3), " (linear rise)")
print("value at 16000 / value at 4000:", round(lrate(16000) / lrate(4000), 3), " (inverse square root fall)")
assert abs(lr.max() - (512 * 4000) ** -0.5) < 1e-12

## 3. Training time and cost (§5.2, §6.1 footnote 5, Table 2)

In [ ]:
print("base: 100,000 steps x 0.4 s =", 100_000 * 0.4 / 3600, "hours")
print("big : 300,000 steps x 1.0 s =", round(300_000 * 1.0 / 86400, 2), "days")
p100 = 9.5e12                                   # sustained FLOPS per P100 (footnote 5)
print(f"base cost estimate: 12 h x 8 GPUs x 9.5 TFLOPS = {12 * 3600 * 8 * p100:.2e} FLOPs (Table 2: 3.3e18)")
print(f"big  cost estimate: 3.5 d x 8 GPUs x 9.5 TFLOPS = {3.5 * 86400 * 8 * p100:.2e} FLOPs (Table 2: 2.3e19)")
t2 = pd.read_csv("data/table2.csv")
print(t2.to_string(index=False))
r = t2.set_index("model")
print("EN-DE: cheapest earlier model with a cost (ConvS2S) / base:", round(r.cost_en_de["ConvS2S"] / r.cost_en_de["Transformer (base model)"], 1))
print("EN-DE: best earlier ensemble cost (ConvS2S Ensemble) / big:", round(r.cost_en_de["ConvS2S Ensemble"] / r.cost_en_de["Transformer (big)"], 1))
print("EN-DE: big BLEU - best earlier (ConvS2S Ensemble):", round(r.bleu_en_de["Transformer (big)"] - r.bleu_en_de["ConvS2S Ensemble"], 2))
print("EN-FR: ConvS2S Ensemble cost / big:", round(r.cost_en_fr["ConvS2S Ensemble"] / r.cost_en_fr["Transformer (big)"], 1))
print("EN-FR: big cost / MoE cost:", round(r.cost_en_fr["Transformer (big)"] / r.cost_en_fr["MoE"], 3))

## 4. Label smoothing (§5.4; Szegedy et al. 2016, §7)
Smoothed target: q'(k) = (1 − ε)·[k = correct] + ε/K. Vocabulary of K = 4 words; correct word "nous"; ε = 0.1.
Model probabilities from the logits 2.0, 1.0, 0.5, −1.0.

In [ ]:
vocab = ["nous", "sommes", "amis", "<end>"]
K, eps, y = len(vocab), 0.1, 0
one_hot = np.eye(K)[y]
smooth = (1 - eps) * one_hot + eps / K
print("one-hot target :", one_hot)
print("smoothed target:", smooth, " sum =", smooth.sum())

def softmax(z): z = np.exp(z - z.max()); return z / z.sum()
def ce(q, p): return -(q * np.log(p)).sum()

cases = {"model (logits 2, 1, 0.5, -1)": softmax(np.array([2.0, 1.0, 0.5, -1.0])),
         "matches smoothed target": smooth,
         "over-confident (0.999)": np.array([0.999] + [0.001 / 3] * 3)}
rows = [(n, np.round(p, 3), round(ce(one_hot, p), 3), round(ce(smooth, p), 3)) for n, p in cases.items()]
print(pd.DataFrame(rows, columns=["prediction", "p", "loss, one-hot", "loss, smoothed"]).to_string(index=False))

# Which confidence on the correct word minimises each loss? (rest spread evenly)
c = np.linspace(0.30, 0.9999, 200_000)
P = np.stack([c] + [(1 - c) / 3] * 3, axis=1)
l_smooth = -(np.log(P) * smooth).sum(1)
print("one-hot loss keeps falling: at c = 0.99 ->", round(-np.log(0.99), 4), ", at c = 0.9999 ->", round(-np.log(0.9999), 4))
print("smoothed loss is lowest at c =", round(c[l_smooth.argmin()], 4), " value", round(l_smooth.min(), 4))
print("smoothed loss at c = 0.9999:", round(l_smooth[-1], 4))

Table 3, rows (D), from the paper (English-to-German development set): label smoothing raises perplexity but BLEU is best at ε = 0.1.

In [ ]:
print(pd.DataFrame({"eps_ls": [0.0, 0.1, 0.2], "PPL (dev)": [4.67, 4.92, 5.47], "BLEU (dev)": [25.3, 25.8, 25.7]}).to_string(index=False))